# Exploring the Latent Space of a Pretrained StyleGAN2 Model

**Objective:** To Understand how moving through the latent space (z-space) of a GAN produces smooth, meaningful changes in generated images.

This uses NVIDIA's official pretrained StyleGAN2-ADA model (trained on FFHQ faces), so you are not training anything from scratch — only using and analyzing an existing model, exactly as the assignment requires.

## 1. Setup: clone the official repo and install dependencies

In [8]:
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch
!pip install ninja imageio -q


## 2. Load the pretrained model

We use NVIDIA's pretrained StyleGAN2-ADA model trained on the FFHQ dataset to generate human face images. The pretrained model allows us to explore the latent space without training a model from scratch.

In [9]:
import pickle, torch, numpy as np
import PIL.Image
import legacy, dnnlib

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

network_pkl = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl'
with dnnlib.util.open_url(network_pkl) as f:
    G = legacy.load_network_pkl(f)['G_ema'].to(device)

print('Model loaded. Latent (z) dimension size:', G.z_dim)


## 3. Helper function to turn a latent vector into an image

In [10]:
def generate_image(G, z, truncation_psi=0.7):
    label = torch.zeros([1, G.c_dim], device=device)
    img = G(z, label, truncation_psi=truncation_psi, noise_mode='const')
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    return PIL.Image.fromarray(img[0].cpu().numpy(), 'RGB')


## 4. Step 1 — Generate two images from two random latent vectors (z1, z2)

A fixed random seed is used so that the generated images can be reproduced. Two random latent vectors, z1 and z2, are generated from the 512-dimensional latent space and converted into images using the pretrained StyleGAN2-ADA model.

In [11]:
SEED = 2005
rng = np.random.RandomState(SEED)

z1 = torch.from_numpy(rng.randn(1, G.z_dim)).to(device).float()
z2 = torch.from_numpy(rng.randn(1, G.z_dim)).to(device).float()

img1 = generate_image(G, z1)
img2 = generate_image(G, z2)

img1.save('z1_image.png')
img2.save('z2_image.png')

display(img1)
display(img2)


## 5. Step 2 — Linear interpolation between z1 and z2

We generate 10 intermediate latent vectors between z1 and z2 using `z = (1 - a) * z1 + a * z2` for a values spaced evenly between 0 and 1, then turn each one into an image.

In [12]:
n_steps = 10
alphas = np.linspace(0, 1, n_steps)

interp_images = []
for i, a in enumerate(alphas):
    z_interp = (1 - a) * z1 + a * z2
    img = generate_image(G, z_interp)
    interp_images.append(img)
    img.save(f'interp_{i:02d}.png')

print(f'Generated {len(interp_images)} interpolation frames.')


## 6. Step 3 — Combine into a single image strip and a GIF

In [13]:
# --- Image strip ---
widths, heights = zip(*(im.size for im in interp_images))
strip = PIL.Image.new('RGB', (sum(widths), max(heights)))
x_offset = 0
for im in interp_images:
    strip.paste(im, (x_offset, 0))
    x_offset += im.size[0]
strip.save('interpolation_strip.png')

# --- GIF ---
interp_images[0].save(
    'interpolation.gif',
    save_all=True,
    append_images=interp_images[1:] + interp_images[::-1],  # forward then reverse, loops smoothly
    duration=180,
    loop=0
)

print('Saved interpolation_strip.png and interpolation.gif')
display(strip)


## 7. Step 4 — Experiment: change only one or two latent dimensions

Here we take z1, leave every dimension untouched except two, and push those two dimensions by a fixed amount. Compare `img1` (original) with `img_mod` (modified) to see what visibly changed.

In [14]:
dims_to_change = [100, 200]
shift_amount = 6.0

z_mod = z1.clone()
for d in dims_to_change:
    z_mod[0, d] += shift_amount

img_mod = generate_image(G, z_mod)
img_mod.save('dimension_experiment.png')

print('Original (z1) vs modified (only dims', dims_to_change, 'changed):')
display(img1)
display(img_mod)


8. Written Observation

The interpolation between z1 and z2 produced a clear, gradual transformation rather than an abrupt change. Features such as hair, background, and overall facial structure changed progressively across the sequence. This demonstrates that linear interpolation in the latent space can produce smooth transitions between generated images.

In the dimension experiment, only dimensions 100 and 200 of z1 were changed by a fixed amount. The resulting image showed relatively little noticeable visual change compared with the original image. This suggests that changing a small number of individual latent dimensions may have a limited visible effect, while the generated image is influenced by the combined effect of many latent dimensions.